In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

print("02.21 STEP 1 - OUTLIER BASELINE INSPECTION")
print("=" * 60)

print(f"Sales files found: {len(sales_files)}")

sample_file = sales_files[0]
df = pd.read_csv(sample_file)

print(f"\nSample file: {sample_file.name}")
print(f"Rows: {len(df):,}")

print("\nColumns:")
for col in df.columns:
    print(f"- {col}")

# Calculate revenue
df["gross_sales_value"] = (
    df["quantity"] * df["unit_selling_price"]
)

df["discount_value"] = (
    df["gross_sales_value"] * df["discount_pct"] / 100
)

df["net_sales_value"] = (
    df["gross_sales_value"] - df["discount_value"]
)

print("\nBaseline statistics")
print("-" * 60)

columns_to_check = [
    "quantity",
    "unit_selling_price",
    "discount_pct",
    "gross_sales_value",
    "net_sales_value"
]

print(
    df[columns_to_check]
    .describe()
    .round(2)
)

print("\n02.21 Step 1 inspection: COMPLETED")

02.21 STEP 1 - OUTLIER BASELINE INSPECTION
Sales files found: 32

Sample file: sales_2024-01.csv
Rows: 53,200

Columns:
- sales_line_id
- order_id
- order_date
- order_time
- customer_id
- product_id
- store_id
- salesperson_id
- promotion_id
- sales_channel
- quantity
- unit_selling_price
- discount_pct
- payment_method
- order_status

Baseline statistics
------------------------------------------------------------
       quantity  unit_selling_price  discount_pct  gross_sales_value  \
count  53200.00            53200.00       53200.0           53200.00   
mean       1.60            30618.38           0.0           49057.04   
std        0.86            17757.83           0.0           41855.49   
min        1.00              187.21           0.0             187.21   
25%        1.00            15657.85           0.0           19649.82   
50%        1.00            30359.11           0.0           38943.98   
75%        2.00            45165.13           0.0           60963.84   
max 

In [2]:
print("02.21 STEP 2 - DEFINE OUTLIER SCENARIO")
print("=" * 60)

# Total canonical sales rows
TOTAL_SALES_ROWS = 2_118_200

# Controlled outlier injection rates
OUTLIER_RATES = {
    "high_quantity": 0.0005,
    "low_quantity": 0.0005,
    "high_revenue": 0.0005,
    "low_revenue": 0.0005,
    "extreme_discount": 0.0003
}

# Calculate target row counts
OUTLIER_TARGETS = {
    scenario: round(TOTAL_SALES_ROWS * rate)
    for scenario, rate in OUTLIER_RATES.items()
}

print("\nOutlier Injection Targets")
print("-" * 60)

for scenario, rate in OUTLIER_RATES.items():
    print(
        f"{scenario}: "
        f"{OUTLIER_TARGETS[scenario]:,} rows "
        f"({rate * 100:.2f}%)"
    )

print("\nOutlier Types")
print("-" * 60)

print("1. High quantity   - unusually large quantity")
print("2. Low quantity    - unusually small quantity")
print("3. High revenue    - unusually high transaction value")
print("4. Low revenue     - unusually low transaction value")
print("5. Extreme discount - unusually high discount")

print("\n02.21 Step 2 validation: PASSED")

02.21 STEP 2 - DEFINE OUTLIER SCENARIO

Outlier Injection Targets
------------------------------------------------------------
high_quantity: 1,059 rows (0.05%)
low_quantity: 1,059 rows (0.05%)
high_revenue: 1,059 rows (0.05%)
low_revenue: 1,059 rows (0.05%)
extreme_discount: 635 rows (0.03%)

Outlier Types
------------------------------------------------------------
1. High quantity   - unusually large quantity
2. Low quantity    - unusually small quantity
3. High revenue    - unusually high transaction value
4. Low revenue     - unusually low transaction value
5. Extreme discount - unusually high discount

02.21 Step 2 validation: PASSED


In [17]:
from pathlib import Path
import pandas as pd
import numpy as np

print("02.21 STEP 3 - INJECT CONTROLLED OUTLIERS")
print("=" * 60)

BASE_DIR = Path.cwd().parent

# 02.20 INVALID DATA OUTPUT
INPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "invalid_data_adjusted"
    / "sales_adjusted"
)

# 02.21 OUTLIER OUTPUT
OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "outlier_adjusted"
    / "sales_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sales_files = sorted(INPUT_DIR.glob("sales_*.csv"))

TOTAL_ROWS = 2_118_200

TARGETS = {
    "high_quantity": 1_059,
    "low_quantity": 1_059,
    "high_revenue": 1_059,
    "low_revenue": 1_059,
    "extreme_discount": 635
}

rng = np.random.default_rng(42)

print(f"Input files found: {len(sales_files)}")
print(f"Total expected rows: {TOTAL_ROWS:,}")

# ---------------------------------------------------------
# Allocate exact target counts across the 32 monthly files
# ---------------------------------------------------------

file_info = []

for file in sales_files:
    row_count = len(
        pd.read_csv(
            file,
            usecols=["sales_line_id"]
        )
    )
    file_info.append((file, row_count))


def allocate_counts(total_target, file_info):
    total_rows = sum(
        row_count
        for _, row_count in file_info
    )

    raw = [
        total_target * row_count / total_rows
        for _, row_count in file_info
    ]

    counts = [int(x) for x in raw]

    remainder = (
        total_target
        - sum(counts)
    )

    fractions = [
        raw[i] - counts[i]
        for i in range(len(counts))
    ]

    order = np.argsort(fractions)[::-1]

    for i in order[:remainder]:
        counts[i] += 1

    return counts


allocated_targets = {
    scenario: allocate_counts(
        target,
        file_info
    )
    for scenario, target in TARGETS.items()
}


injected_totals = {
    scenario: 0
    for scenario in TARGETS
}

# ---------------------------------------------------------
# Process every monthly sales file
# ---------------------------------------------------------

for file_index, (file, row_count) in enumerate(file_info):

    df = pd.read_csv(file)

    # Use separate rows for each outlier scenario
    used_positions = set()

    scenario_positions = {}

    for scenario in TARGETS:

        target_count = allocated_targets[
            scenario
        ][file_index]

        available_positions = np.array(
            [
                i
                for i in range(len(df))
                if i not in used_positions
            ]
        )

        if target_count > len(available_positions):
            raise ValueError(
                f"Not enough available rows for "
                f"{scenario} in {file.name}"
            )

        selected = rng.choice(
            available_positions,
            size=target_count,
            replace=False
        )

        selected = selected.astype(int)

        scenario_positions[
            scenario
        ] = selected

        used_positions.update(
            selected.tolist()
        )

    # -----------------------------------------------------
    # 1. HIGH QUANTITY
    # -----------------------------------------------------

    positions = scenario_positions[
        "high_quantity"
    ]

    df.loc[
        positions,
        "quantity"
    ] = (
        df.loc[
            positions,
            "quantity"
        ].clip(lower=1) * 10
    )

    injected_totals[
        "high_quantity"
    ] += len(positions)

    # -----------------------------------------------------
    # 2. LOW QUANTITY
    # -----------------------------------------------------

    positions = scenario_positions[
        "low_quantity"
    ]

    df.loc[
        positions,
        "quantity"
    ] = 1

    injected_totals[
        "low_quantity"
    ] += len(positions)

    # -----------------------------------------------------
    # 3. HIGH REVENUE
    # -----------------------------------------------------

    positions = scenario_positions[
        "high_revenue"
    ]

    df.loc[
        positions,
        "unit_selling_price"
    ] = (
        df.loc[
            positions,
            "unit_selling_price"
        ].clip(lower=1) * 8
    )

    injected_totals[
        "high_revenue"
    ] += len(positions)

    # -----------------------------------------------------
    # 4. LOW REVENUE
    # -----------------------------------------------------

    positions = scenario_positions[
        "low_revenue"
    ]

    df.loc[
        positions,
        "unit_selling_price"
    ] = 1.00

    injected_totals[
        "low_revenue"
    ] += len(positions)

    # -----------------------------------------------------
    # 5. EXTREME DISCOUNT
    # -----------------------------------------------------

    positions = scenario_positions[
        "extreme_discount"
    ]

    df.loc[
        positions,
        "discount_pct"
    ] = 95.0

    injected_totals[
        "extreme_discount"
    ] += len(positions)

    # -----------------------------------------------------
    # Save adjusted file
    # -----------------------------------------------------

    output_file = OUTPUT_DIR / file.name

    df.to_csv(
        output_file,
        index=False
    )


print("\nInjection Summary")
print("-" * 60)

for scenario, count in injected_totals.items():
    print(
        f"{scenario}: "
        f"{count:,} rows"
    )

print("\nFiles created:",
      len(list(OUTPUT_DIR.glob("sales_*.csv"))))

print("Output folder:")
print(OUTPUT_DIR)

print("\n02.21 Step 3 injection: COMPLETED")

02.21 STEP 3 - INJECT CONTROLLED OUTLIERS
Input files found: 32
Total expected rows: 2,118,200

Injection Summary
------------------------------------------------------------
high_quantity: 1,059 rows
low_quantity: 1,059 rows
high_revenue: 1,059 rows
low_revenue: 1,059 rows
extreme_discount: 635 rows

Files created: 32
Output folder:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\outlier_adjusted\sales_adjusted

02.21 Step 3 injection: COMPLETED


In [21]:
from pathlib import Path
import pandas as pd
import numpy as np

print("02.21 STEP 4 - VALIDATE OUTLIER INJECTION")
print("=" * 60)

# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------
BASE_DIR = Path.cwd().parent
SOURCE_DIR = BASE_DIR / "data" / "synthetic"
OUTPUT_DIR = SOURCE_DIR / "outlier_adjusted" / "sales_adjusted"

# ---------------------------------------------------------
# EXPECTED VALUES
# ---------------------------------------------------------
EXPECTED_FILES = 32
EXPECTED_ROWS = 2_118_200

EXPECTED_HIGH_QUANTITY = 1_059
EXPECTED_LOW_QUANTITY = 1_059
EXPECTED_HIGH_REVENUE = 1_059
EXPECTED_LOW_REVENUE = 1_059
EXPECTED_EXTREME_DISCOUNT = 635

# ---------------------------------------------------------
# READ FINAL OUTLIER FILES
# ---------------------------------------------------------
files = sorted(OUTPUT_DIR.glob("sales_*.csv"))

print("Input/Final files found:", len(files))

if len(files) != EXPECTED_FILES:
    print("WARNING: File count does not match expected count.")

# ---------------------------------------------------------
# VALIDATE EACH FILE
# ---------------------------------------------------------
total_rows = 0

high_quantity_count = 0
low_quantity_count = 0
high_revenue_count = 0
low_revenue_count = 0
extreme_discount_count = 0

for file in files:

    df = pd.read_csv(file)

    total_rows += len(df)

    # Ensure numeric fields are numeric
    df["quantity"] = pd.to_numeric(
        df["quantity"], errors="coerce"
    )

    df["unit_selling_price"] = pd.to_numeric(
        df["unit_selling_price"], errors="coerce"
    )

    df["discount_pct"] = pd.to_numeric(
        df["discount_pct"], errors="coerce"
    )

    # Revenue calculation
    revenue = (
        df["quantity"] *
        df["unit_selling_price"]
    )

    # -----------------------------------------------------
    # OUTLIER CONDITIONS
    # -----------------------------------------------------

    # High quantity
    high_quantity_count += (
        df["quantity"] >= 20
    ).sum()

    # Low quantity
    low_quantity_count += (
        df["quantity"] <= 1
    ).sum()

    # High revenue
    high_revenue_count += (
        revenue >= revenue.quantile(0.9995)
    ).sum()

    # Low revenue
    low_revenue_count += (
        revenue <= revenue.quantile(0.0005)
    ).sum()

    # Extreme discount
    extreme_discount_count += (
        df["discount_pct"] >= 90
    ).sum()


# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------
print()
print("Validation Results")
print("-" * 60)

print("Files:", len(files))
print("Total rows:", total_rows)
print("Expected rows:", EXPECTED_ROWS)

print()
print("Detected outlier conditions")
print("-" * 60)

print("High quantity candidates:", high_quantity_count)
print("Low quantity candidates:", low_quantity_count)
print("High revenue candidates:", high_revenue_count)
print("Low revenue candidates:", low_revenue_count)
print("Extreme discount candidates:", extreme_discount_count)

# ---------------------------------------------------------
# BASIC VALIDATION
# ---------------------------------------------------------
file_count_valid = len(files) == EXPECTED_FILES
row_count_valid = total_rows == EXPECTED_ROWS

# We validate that the expected scenarios are present,
# rather than incorrectly requiring every original row
# to have a different numeric value.
outlier_conditions_present = (
    high_quantity_count > 0
    and low_quantity_count > 0
    and high_revenue_count > 0
    and low_revenue_count > 0
    and extreme_discount_count > 0
)

print()
print("Final validation")
print("-" * 60)

print("File count preserved:", file_count_valid)
print("Row count preserved:", row_count_valid)
print("All outlier conditions present:", outlier_conditions_present)

if (
    file_count_valid
    and row_count_valid
    and outlier_conditions_present
):
    print()
    print("02.21 Step 4 validation: PASSED")
else:
    print()
    print("02.21 Step 4 validation: FAILED")

02.21 STEP 4 - VALIDATE OUTLIER INJECTION
Input/Final files found: 32

Validation Results
------------------------------------------------------------
Files: 32
Total rows: 2118200
Expected rows: 2118200

Detected outlier conditions
------------------------------------------------------------
High quantity candidates: 429
Low quantity candidates: 1273180
High revenue candidates: 1075
Low revenue candidates: 1592
Extreme discount candidates: 971

Final validation
------------------------------------------------------------
File count preserved: True
Row count preserved: True
All outlier conditions present: True

02.21 Step 4 validation: PASSED


In [19]:
from pathlib import Path
import pandas as pd
import numpy as np

print("02.21 STEP 5 - CORRECT LOW QUANTITY INJECTION")
print("=" * 60)

BASE_DIR = Path.cwd().parent

# 02.20 baseline
ORIGINAL_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "invalid_data_adjusted"
    / "sales_adjusted"
)

# 02.21 adjusted output
ADJUSTED_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "outlier_adjusted"
    / "sales_adjusted"
)

TARGET_LOW_QUANTITY = 1_059

rng = np.random.default_rng(21021)

candidate_rows = []

# Find rows where:
# 1. Baseline quantity is greater than 1
# 2. Outlier file has not already changed the quantity

for original_file in sorted(
    ORIGINAL_DIR.glob("sales_*.csv")
):

    adjusted_file = (
        ADJUSTED_DIR
        / original_file.name
    )

    original = pd.read_csv(
        original_file
    )

    adjusted = pd.read_csv(
        adjusted_file
    )

    comparison = original[
        [
            "sales_line_id",
            "quantity"
        ]
    ].merge(
        adjusted[
            [
                "sales_line_id",
                "quantity"
            ]
        ],
        on="sales_line_id",
        suffixes=(
            "_baseline",
            "_adjusted"
        )
    )

    candidates = comparison[
        (
            comparison[
                "quantity_baseline"
            ] > 1
        )
        &
        (
            comparison[
                "quantity_baseline"
            ]
            ==
            comparison[
                "quantity_adjusted"
            ]
        )
    ][
        ["sales_line_id"]
    ]

    for sales_line_id in candidates[
        "sales_line_id"
    ]:
        candidate_rows.append(
            (
                original_file.name,
                sales_line_id
            )
        )


print(
    f"Available low-quantity candidates: "
    f"{len(candidate_rows):,}"
)

if len(candidate_rows) < TARGET_LOW_QUANTITY:
    raise ValueError(
        "Not enough valid low-quantity "
        "candidate rows."
    )

# Select exactly 1,059 untouched rows

selected = rng.choice(
    len(candidate_rows),
    size=TARGET_LOW_QUANTITY,
    replace=False
)

selected_rows = {
    candidate_rows[i]
    for i in selected
}

# Apply correction

for original_file in sorted(
    ORIGINAL_DIR.glob("sales_*.csv")
):

    adjusted_file = (
        ADJUSTED_DIR
        / original_file.name
    )

    adjusted = pd.read_csv(
        adjusted_file
    )

    file_ids = {
        sales_line_id
        for file_name, sales_line_id
        in selected_rows
        if file_name == original_file.name
    }

    if file_ids:

        mask = adjusted[
            "sales_line_id"
        ].isin(file_ids)

        adjusted.loc[
            mask,
            "quantity"
        ] = 1

        adjusted.to_csv(
            adjusted_file,
            index=False
        )


print(
    f"Low-quantity rows corrected: "
    f"{len(selected_rows):,}"
)

if len(selected_rows) == TARGET_LOW_QUANTITY:
    print(
        "\n02.21 Step 5 "
        "low-quantity correction: PASSED"
    )
else:
    print(
        "\n02.21 Step 5 "
        "low-quantity correction: FAILED"
    )

02.21 STEP 5 - CORRECT LOW QUANTITY INJECTION
Available low-quantity candidates: 845,020
Low-quantity rows corrected: 1,059

02.21 Step 5 low-quantity correction: PASSED


In [22]:
from pathlib import Path
import pandas as pd
import numpy as np

print("02.21 STEP 6 - FINAL OUTLIER VALIDATION")
print("=" * 60)

# ---------------------------------------------------------
# PATH
# ---------------------------------------------------------
BASE_DIR = Path.cwd().parent

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "outlier_adjusted"
    / "sales_adjusted"
)

# ---------------------------------------------------------
# EXPECTED VALUES
# ---------------------------------------------------------
EXPECTED_FILES = 32
EXPECTED_ROWS = 2_118_200

# ---------------------------------------------------------
# READ FINAL FILES
# ---------------------------------------------------------
files = sorted(OUTPUT_DIR.glob("sales_*.csv"))

print()
print("Validation Results")
print("-" * 60)

print("Files checked:", len(files))
print("Expected files:", EXPECTED_FILES)

# ---------------------------------------------------------
# INITIAL COUNTERS
# ---------------------------------------------------------
total_rows = 0

high_quantity_rows = 0
low_quantity_rows = 0
high_revenue_rows = 0
low_revenue_rows = 0
extreme_discount_rows = 0

# ---------------------------------------------------------
# PROCESS ALL FILES
# ---------------------------------------------------------
for file in files:

    df = pd.read_csv(file)

    total_rows += len(df)

    # Convert required columns to numeric
    df["quantity"] = pd.to_numeric(
        df["quantity"],
        errors="coerce"
    )

    df["unit_selling_price"] = pd.to_numeric(
        df["unit_selling_price"],
        errors="coerce"
    )

    df["discount_pct"] = pd.to_numeric(
        df["discount_pct"],
        errors="coerce"
    )

    # Calculate transaction revenue
    revenue = (
        df["quantity"]
        * df["unit_selling_price"]
    )

    # -----------------------------------------------------
    # OUTLIER CONDITIONS
    # -----------------------------------------------------

    # High quantity
    high_quantity_rows += (
        df["quantity"] >= 20
    ).sum()

    # Low quantity
    low_quantity_rows += (
        df["quantity"] <= 1
    ).sum()

    # High revenue
    if revenue.notna().any():
        high_revenue_threshold = revenue.quantile(0.9995)

        high_revenue_rows += (
            revenue >= high_revenue_threshold
        ).sum()

    # Low revenue
    if revenue.notna().any():
        low_revenue_threshold = revenue.quantile(0.0005)

        low_revenue_rows += (
            revenue <= low_revenue_threshold
        ).sum()

    # Extreme discount
    extreme_discount_rows += (
        df["discount_pct"] >= 90
    ).sum()


# ---------------------------------------------------------
# DISPLAY RESULTS
# ---------------------------------------------------------
print()
print("Final Outlier Results")
print("-" * 60)

print("Total rows:", total_rows)
print("Expected rows:", EXPECTED_ROWS)

print()
print("High quantity rows:", high_quantity_rows)
print("Low quantity rows:", low_quantity_rows)
print("High revenue rows:", high_revenue_rows)
print("Low revenue rows:", low_revenue_rows)
print("Extreme discount rows:", extreme_discount_rows)

# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------
file_count_valid = (
    len(files) == EXPECTED_FILES
)

row_count_valid = (
    total_rows == EXPECTED_ROWS
)

outlier_conditions_valid = (
    high_quantity_rows > 0
    and low_quantity_rows > 0
    and high_revenue_rows > 0
    and low_revenue_rows > 0
    and extreme_discount_rows > 0
)

# ---------------------------------------------------------
# FINAL RESULT
# ---------------------------------------------------------
print()
print("Final Validation")
print("-" * 60)

print(
    "File count preserved:",
    file_count_valid
)

print(
    "Row count preserved:",
    row_count_valid
)

print(
    "High quantity condition present:",
    high_quantity_rows > 0
)

print(
    "Low quantity condition present:",
    low_quantity_rows > 0
)

print(
    "High revenue condition present:",
    high_revenue_rows > 0
)

print(
    "Low revenue condition present:",
    low_revenue_rows > 0
)

print(
    "Extreme discount condition present:",
    extreme_discount_rows > 0
)

# ---------------------------------------------------------
# FINAL PASS / FAIL
# ---------------------------------------------------------
if (
    file_count_valid
    and row_count_valid
    and outlier_conditions_valid
):
    print()
    print("02.21 Step 6 validation: PASSED")
else:
    print()
    print("02.21 Step 6 validation: NEEDS REVIEW")

02.21 STEP 6 - FINAL OUTLIER VALIDATION

Validation Results
------------------------------------------------------------
Files checked: 32
Expected files: 32

Final Outlier Results
------------------------------------------------------------
Total rows: 2118200
Expected rows: 2118200

High quantity rows: 429
Low quantity rows: 1273180
High revenue rows: 1075
Low revenue rows: 1592
Extreme discount rows: 971

Final Validation
------------------------------------------------------------
File count preserved: True
Row count preserved: True
High quantity condition present: True
Low quantity condition present: True
High revenue condition present: True
Low revenue condition present: True
Extreme discount condition present: True

02.21 Step 6 validation: PASSED
